# Explore & Ingest — Is My Stuff Safe?

Fetch a sample of CPSC recall data, inspect the raw JSON shape, and try out ingestion, hybrid search, and the full agent before wiring up the app.

In [ ]:
import sys
sys.path.append('..')

import requests

resp = requests.get(
    'https://www.saferproducts.gov/RestWebServices/Recall',
    params={'format': 'json', 'ProductName': 'stroller'}
)
data = resp.json()
print('Number of records:', len(data))
data[0]

## Fetch the full dataset (all categories)

Run this once — it writes to `../data/recalls_raw.json`.

In [ ]:
!python ../data/fetch_data.py

## Build documents + keyword & vector indices

In [ ]:
from ingest import load_documents, build_index, build_vector_index

documents = load_documents()
print(f'Loaded {len(documents)} documents')
documents[0]

In [ ]:
index = build_index(documents)
vector_index = build_vector_index(documents)  # calls OpenAI embeddings API

print('Keyword search:')
for r in index.search('stroller wheel falling off', num_results=5):
    print('-', r['title'][:100])

print('\nVector search:')
for r in vector_index.search('my kid\'s sleeper thing is dangerous', num_results=5):
    print('-', r['title'][:100])

## Try hybrid search + the full RAG answer

In [ ]:
from rag_helper import RAGBase

rag = RAGBase(index=index, vector_index=vector_index, retrieval_mode='hybrid')
result = rag.answer('Have there been any recalls for Fisher-Price baby swings?')
print(result['answer'])

## Try the full agent (with tool calling)

In [ ]:
from tools import RecallTools
from agent import RecallAgent

tools = RecallTools(rag=rag, documents=documents)
agent = RecallAgent(recall_tools=tools)

result = agent.ask('I have a Fisher-Price rock-n-play from around 2018, is it safe?', verbose=True)
print(result['answer'])

## Run evaluation

These write results to the `evaluation/` folder. See `evaluation/evaluate.py` and `evaluation/llm_eval.py` for details.

In [ ]:
!python ../evaluation/ground_truth.py
!python ../evaluation/evaluate.py
!python ../evaluation/llm_eval.py